# Cellpose-SAM Segmentation on Xenium Images

This notebook runs Cellpose-SAM segmentation on both the morphology
(nuclear stain) and H&E images, and visualizes the results.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from cpsam_xenium_analysis import config, data_loader as dl
from cpsam_xenium_analysis.segmentation import CPSAMSegmentor, filter_masks
from cpsam_xenium_analysis.visualization import plots as vis

%matplotlib inline


## 1. Load Images (Cropped ROI)


In [ ]:
crop = config.CROP_ROI
morph_img = dl.load_morphology_image(use_focus=True, crop_roi=crop)
he_img = dl.load_he_image(crop_roi=crop)
print(f'Morphology: {morph_img.shape}')
print(f'H&E: {he_img.shape}')


## 2. Run Cellpose-SAM on Morphology Image


In [ ]:
# Initialize segmentor
segmentor = CPSAMSegmentor(gpu=True, use_bfloat16=True)

# Run segmentation on morphology image
mask_morph, flows_morph, styles_morph = segmentor.segment_morphology(morph_img)
mask_morph = filter_masks(mask_morph, min_size=15)

n_cells = len(np.unique(mask_morph)) - 1
print(f'Found {n_cells} cells in morphology image')


In [ ]:
# Visualize segmentation overlay
fig = vis.plot_segmentation_overlay(
    morph_img, mask_morph,
    title=f'Cellpose-SAM on Morphology ({n_cells} cells)'
)


## 3. Run Cellpose-SAM on H&E Image


In [ ]:
mask_he, flows_he, styles_he = segmentor.segment_he(he_img)
mask_he = filter_masks(mask_he, min_size=15)

n_cells_he = len(np.unique(mask_he)) - 1
print(f'Found {n_cells_he} cells in H&E image')


In [ ]:
# Visualize H&E segmentation
fig = vis.plot_segmentation_overlay(
    he_img, mask_he,
    title=f'Cellpose-SAM on H&E ({n_cells_he} cells)',
    outline_color='yellow',
)


## 4. Side-by-Side Comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(morph_img)
axes[0].imshow((mask_morph > 0).astype(float), cmap='jet', alpha=0.3)
axes[0].set_title(f'Morphology + CPSAM ({n_cells} cells)')
axes[0].axis('off')

axes[1].imshow(he_img)
axes[1].imshow((mask_he > 0).astype(float), cmap='jet', alpha=0.3)
axes[1].set_title(f'H&E + CPSAM ({n_cells_he} cells)')
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 5. Save Masks for Downstream Analysis


In [ ]:
np.save(config.OUTPUT_DIR / 'masks_cpsam_morphology.npy', mask_morph)
np.save(config.OUTPUT_DIR / 'masks_cpsam_he.npy', mask_he)
print('Masks saved!')
